In [ ]:
#2 simulated datasets
import neuroimage_analysis as na
import numpy as np
import matplotlib.pyplot as plt
import os 
import glob
import nibabel as nib
from tqdm import tqdm
from scipy.stats import rankdata, pearsonr

# Directories

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0

# Ground truth Schaefer 300 FC maps
schaefer_dir = os.path.join(dir, "data/schaefer_300_fcmap")
schaefer_files = sorted(glob.glob(os.path.join(schaefer_dir, '*.nii.gz')))
print(f'Found {len(schaefer_files)} Schaefer region FC maps for ground truth')

# Simulated lesion FC maps
lesion_dir = os.path.join(dir, "data/sim_lesions")
lesion_files = sorted(glob.glob(os.path.join(lesion_dir, 'lesion*AvgR.nii.gz')))
print(f'Found {len(lesion_files)} simulated lesion FC maps')

outdir = os.path.join(dir, "results")


# Main Analysis 2: Simulated Datasets

Here, each simulated ‘study’ compared sLNM maps calculated from two separately simulated datasets with distinct ground truth symptom networks.

**Simulated lesions**
Lesions are 4-mm radius spheres placed at randomly sampled brain voxels. A pool of 500 lesion FC maps is generated in advance; for each simulated dataset, 100 subjects are bootstrapped (with replacement) from this pool. 

**Modeling clinical symptoms**
A ground truth network is defined by randomly selecting one of 300 Schaefer parcels; its whole-brain FC map defines the ground truth network, which represents the "true" disease network for their respective dataset. Symptoms ($S$) are modeled as a linear function of the spatial correlation between the lesion FC map and the ground truth network ($R$), plus noise ($\epsilon$):

$$S = \theta R + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

The effect size parameter $\eta^2$ controls the proportion of symptom variance explained by the ground truth network:

$$\eta^2 = \frac{\theta^2}{\theta^2 + \sigma^2}$$

We test three levels of effect size: $\eta^2 = 0.0$, $0.3$, and $0.99$.

**Convergence test (symptoms-permutation test)**
For each pair of simulated datasets, we compute sLNM maps and test whether their spatial similarity exceeds a permutation-based null distribution.

In [ ]:
def gen_dataset(subject_maps, ground_truth_maps, sample_size=100, effect_size=0.3,
                ground_truth_seed=None, rank=False):
    """
    Generate a simulated dataset with synthetic behavioral scores.

    Bootstraps subjects from lesion FC maps and creates behavioral scores
    by correlating each subject's FC map with a ground truth Schaefer region map,
    scaled by a specified effect size with added noise.

    Parameters
    ----------
    subject_maps      : list of paths to subject FC NIfTI files
    ground_truth_maps : list of paths to Schaefer region FC NIfTI files
    sample_size       : number of subjects to bootstrap (default: 100)
    effect_size       : strength of brain-behavior relationship (default: 0.3)
    ground_truth_seed : index of ground truth map to use (random if None)
    rank              : if True, rank-transform the brain-behavior correlations

    Returns
    -------
    dict with 'subject_maps' (n_subjects x voxels), 'ground_truth' (voxels,), 'scores' (n_subjects,)
    """
    # Bootstrap subjects
    subject_indices = np.random.choice(len(subject_maps), size=sample_size, replace=True)
    subject_fc_array = np.array([na.nifti_getdata(subject_maps[i]) for i in subject_indices])

    # Interpolate any NaN voxels using neighboring voxel values
    nan_mask = np.isnan(subject_fc_array)
    if np.any(nan_mask):
        for i, j in zip(*np.where(nan_mask)):
            neighbors = []
            if j > 0 and not np.isnan(subject_fc_array[i, j-1]):
                neighbors.append(subject_fc_array[i, j-1])
            if j < subject_fc_array.shape[1]-1 and not np.isnan(subject_fc_array[i, j+1]):
                neighbors.append(subject_fc_array[i, j+1])
            if neighbors:
                subject_fc_array[i, j] = np.mean(neighbors)

    # Select ground truth Schaefer region map
    if ground_truth_seed is None:
        ground_truth_seed = np.random.randint(len(ground_truth_maps))
    ground_truth_map = na.nifti_getdata(ground_truth_maps[ground_truth_seed])

    # Correlate each subject's FC map with the ground truth map
    r_ground_truth = na.pearson_rows(subject_fc_array, ground_truth_map)
    if rank:
        from scipy.stats import rankdata
        r_ground_truth = rankdata(r_ground_truth) # optional analysis

    # Scale by effect size, add noise, and z-score to produce synthetic behavioral scores
    r_scaled = r_ground_truth * np.sqrt(effect_size / (1 - effect_size))
    noise = np.random.normal(0, 1.0, size=subject_fc_array.shape[0])
    scores = r_scaled + noise
    scores = (scores - np.mean(scores)) / np.std(scores)

    return {
        'subject_maps': subject_fc_array,
        'ground_truth': ground_truth_map,
        'scores': scores,
        'ground_truth_seed': ground_truth_seed
    }


def permute_network(A_fcmaps, B_fcmaps, A_outcomes, B_outcomes, n_permutations=1000):
    """
    Generate permuted sLNM maps for two datasets.

    For each permutation, behavioral outcomes are randomly shuffled before
    computing the sLNM map. The first permutation always uses the real outcomes.

    Returns
    -------
    dict with 'A_permuted' and 'B_permuted', each shaped (n_permutations x voxels)
    """
    def make_permutation_matrix(outcomes, n_perms):
        # First column is the real outcome; remaining columns are shuffled
        n = len(outcomes)
        matrix = np.zeros((n, n_perms))
        matrix[:, 0] = outcomes
        for i in range(1, n_perms):
            matrix[:, i] = np.random.permutation(outcomes)
        return matrix

    A_perm_matrix = make_permutation_matrix(A_outcomes, n_permutations)
    B_perm_matrix = make_permutation_matrix(B_outcomes, n_permutations)

    A_permuted_maps = na.voxel_outcome_correlation(A_fcmaps, A_perm_matrix)
    B_permuted_maps = na.voxel_outcome_correlation(B_fcmaps, B_perm_matrix)

    return {
        'A_permuted': A_permuted_maps,
        'B_permuted': B_permuted_maps
    }


def convergence_test(dataset_A, dataset_B, n_permutations=1000):
    """
    Test whether two sLNM maps converge more than expected by chance.

    Computes the spatial similarity (Pearson r) between sLNM maps from two 
    datasets and compares it against a null distribution generated by 
    permuting behavioral outcomes.

    The ground truth r (correlation between the two Schaefer seed maps) is 
    also returned as a baseline — reflecting what the sLNM similarity should 
    be if the method correctly recovered the underlying brain networks.

    Returns
    -------
    dict with 'ground_truth_r', 'empirical_r', 'permuted_r', 'p_value'
    """
    A_fcmaps      = dataset_A['subject_maps']
    B_fcmaps      = dataset_B['subject_maps']
    A_scores      = dataset_A['scores']
    B_scores      = dataset_B['scores']
    A_ground_truth = dataset_A['ground_truth']
    B_ground_truth = dataset_B['ground_truth']

    # Baseline: similarity between the two ground truth Schaefer seed maps
    ground_r = pearsonr(A_ground_truth, B_ground_truth).statistic

    # Compute sLNM maps for each dataset
    A_slnm = na.voxel_outcome_correlation(A_fcmaps, A_scores[:, None]).flatten()
    B_slnm = na.voxel_outcome_correlation(B_fcmaps, B_scores[:, None]).flatten()

    # Empirical sLNM similarity
    emp_r = pearsonr(A_slnm, B_slnm).statistic

    # Null distribution via behavioral permutation
    permuted = permute_network(A_fcmaps, B_fcmaps, A_scores, B_scores, n_permutations)
    permuted_r = np.array([
        pearsonr(permuted['A_permuted'][i, :], permuted['B_permuted'][i, :]).statistic
        for i in range(n_permutations)
    ])

    # p-value: proportion of permuted r values >= empirical r
    p_value = (np.sum(permuted_r >= emp_r) + 1) / (n_permutations + 1)

    return {
        'ground_truth_r': ground_r,
        'empirical_r': emp_r,
        'permuted_r': permuted_r,
        'p_value': p_value
    }

In [ ]:
# Example: test convergence between two simulated datasets
# Dataset A and B are generated from the same lesion FC maps but with different ground truth networks


dataset_A = gen_dataset(
    subject_maps=lesion_files,
    ground_truth_maps=schaefer_files,
    sample_size=100,
    effect_size=0.3
)

dataset_B = gen_dataset(
    subject_maps=lesion_files,
    ground_truth_maps=schaefer_files,
    sample_size=100,
    effect_size=0.3
)

print(f"Dataset A ground truth: Schaefer region {dataset_A['ground_truth_seed']}")
print(f"Dataset B ground truth: Schaefer region {dataset_B['ground_truth_seed']}")

result = convergence_test(dataset_A, dataset_B, n_permutations=1000)

print(f"\nGround truth similarity (expected r): {result['ground_truth_r']:.3f}")
print(f"Empirical sLNM similarity:            {result['empirical_r']:.3f}")
print(f"Permutation p-value:                  {result['p_value']:.3f}")

# Mass Simulations

In [ ]:
# ── Tunable parameters ────────────────────────────────────────────────────────
n_runs      = 1000  # number of simulation runs (recommended at > 1000)
effect_size = 0.3   # eta-squared effect size (brain-behavior relationship strength)
n_perms     = 1000   # permutations per convergence test
sample_size = 100    # subjects per dataset
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
run_results = {
    'ground_truth_similarity': [],
    'slnm_similarity': [],
    'p_values': [],
    'is_significant': []
}

for run in tqdm(range(n_runs), desc='Running simulations...'):
    seed_A, seed_B = np.random.choice(len(schaefer_files), size=2, replace=False)

    dataset_A = gen_dataset(
        subject_maps=lesion_files,
        ground_truth_maps=schaefer_files,
        sample_size=sample_size,
        effect_size=effect_size,
        ground_truth_seed=seed_A
    )
    dataset_B = gen_dataset(
        subject_maps=lesion_files,
        ground_truth_maps=schaefer_files,
        sample_size=sample_size,
        effect_size=effect_size,
        ground_truth_seed=seed_B
    )

    result = convergence_test(dataset_A, dataset_B, n_permutations=n_perms)

    run_results['ground_truth_similarity'].append(result['ground_truth_r'])
    run_results['slnm_similarity'].append(result['empirical_r'])
    run_results['p_values'].append(result['p_value'])
    run_results['is_significant'].append(result['p_value'] < 0.05)

import pickle

pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(run_results, f)

print(f'Saved: {pkl_path}')
print(f'Significant rate: {np.mean(run_results["is_significant"]):.2%}')

In [ ]:
import pickle

pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

ground_r   = np.array(data['ground_truth_similarity'])
slnm_r     = np.array(data['slnm_similarity'])
is_sig     = np.array(data['is_significant'])
p_vals     = np.array(data['p_values'])

bins        = np.arange(-1.0, 1.1, 0.1)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_indices = np.digitize(ground_r, bins)

sig_rates, mean_slnm_r, sem_slnm_r = [], [], []
for i in range(1, len(bins)):
    in_bin = bin_indices == i
    n = np.sum(in_bin)
    sig_rates.append(np.sum(is_sig[in_bin]) / n if n > 0 else np.nan)
    mean_slnm_r.append(np.mean(slnm_r[in_bin]) if n > 0 else np.nan)
    sem_slnm_r.append(np.std(slnm_r[in_bin] / np.sqrt(n)) if n > 0 else np.nan)

sig_rates   = np.array(sig_rates)
mean_slnm_r = np.array(mean_slnm_r)
sem_slnm_r  = np.array(sem_slnm_r)

plt.rcParams['font.size'] = 14
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Proportion of simulations passing p < 0.05
axes[0].bar(bin_centers, sig_rates, width=0.09, color='#7FAFD4', edgecolor='black', alpha=0.8)
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='α = 0.05')
axes[0].set_xlabel('Ground Truth Spatial r')
axes[0].set_ylabel('Proportion of Simulations Passing p < 0.05')
axes[0].set_ylim(0,1.1)
axes[0].set_title('Significant Rate by\nGround Truth Similarity')
axes[0].legend()

# Plot 2: Mean sLNM r by ground truth similarity bin with error bars
axes[1].bar(bin_centers, mean_slnm_r, width=0.09, color='#F5C566', edgecolor='black', alpha=0.8,
            yerr=sem_slnm_r, capsize=3, error_kw={'linewidth': 1.5})
axes[1].plot([-1, 1], [-1, 1], 'k--', alpha=0.5, label='x=y')
axes[1].set_xlabel('Ground Truth Spatial r')
axes[1].set_ylabel('Mean sLNM Spatial r')
axes[1].set_title('sLNM Similarity by\nGround Truth Similarity')
axes[1].legend()

# Plot 3: QQ plot
obs_p = np.sort(p_vals)
exp_p = np.linspace(0, 1, len(p_vals))
axes[2].scatter(exp_p, obs_p, alpha=0.6, s=35, color='#E8846B', label=f'η² = {effect_size}')
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[2].set_xlabel('Expected P-values')
axes[2].set_ylabel('Observed P-values')
axes[2].set_title('QQ Plot')
axes[2].legend(title='Effect Size')

plt.tight_layout()
plt.savefig(os.path.join(outdir, f'simulation_results_eta_{effect_size}.svg'), format='svg', bbox_inches='tight')
plt.show()

print(f'Significant rate: {np.mean(is_sig):.2%} across {n_runs} runs (for one sided test)')

# Comparing All Effect Sizes 

In [ ]:
import pickle

effect_sizes = [0.0, 0.3, 0.99]
n_runs = 1000
colors       = ['#E8846B', '#F5C566', '#7FAFD4']

bins        = np.arange(-1.0, 1.1, 0.1)
bin_centers = (bins[:-1] + bins[1:]) / 2

sig_rates_all = {}
lnm_sim_all   = {}
lnm_sem_all   = {}
target_files  = {}

for effect_size in effect_sizes:
    pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
    target_files[effect_size] = pkl_path
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    ground_r = np.array(data['ground_truth_similarity'])
    slnm_r   = np.array(data['slnm_similarity'])
    is_sig   = np.array(data['is_significant'])

    bin_indices = np.digitize(ground_r, bins)
    sig_rates, mean_slnm_r, sem_slnm_r = [], [], []
    for i in range(1, len(bins)):
        in_bin = bin_indices == i
        n = np.sum(in_bin)
        sig_rates.append(np.sum(is_sig[in_bin]) / n if n > 0 else np.nan)
        mean_slnm_r.append(np.mean(slnm_r[in_bin]) if n > 0 else np.nan)
        sem_slnm_r.append(np.std(slnm_r[in_bin]) / np.sqrt(n) if n > 0 else np.nan)

    sig_rates_all[effect_size] = np.array(sig_rates)
    lnm_sim_all[effect_size]   = np.array(mean_slnm_r)
    lnm_sem_all[effect_size]   = np.array(sem_slnm_r)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

LABEL_SIZE = 12
TICK_SIZE  = 10
LEGEND_SIZE = 10

mask = (bin_centers >= -0.6) & (bin_centers <= 0.6)
bin_centers_cut = bin_centers[mask]
x = np.arange(len(bin_centers_cut))
width = 0.25
bin_labels = [f'{c-0.05:.1f} to {c+0.05:.1f}' for c in bin_centers_cut]

# Panel A: Significant Rate
ax = axes[0]
for i, effect in enumerate(effect_sizes):
    offset = (i - 1) * width
    rates_cut = sig_rates_all[effect][mask]
    ax.bar(x + offset, rates_cut, width, label=f'{effect:.2f}',
           color=colors[i], edgecolor='black', linewidth=0.5)
ax.axhline(y=0.05, color='red', linestyle='--')
ax.set_xlabel('Spatial r of Ground Truth Network', fontsize=LABEL_SIZE)
ax.set_ylabel('Proportion of Simulations with p < 0.05', fontsize=LABEL_SIZE)
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, fontsize=TICK_SIZE, rotation=45, ha='right')
ax.tick_params(axis='y', labelsize=TICK_SIZE)
ax.set_ylim(0.0, 1.0)
ax.grid(alpha=0.2)

handles = [plt.Rectangle((0,0),1,1, facecolor=c, edgecolor='black', linewidth=0.5) for c in colors]
labels  = ['0.00', '0.30', '0.99']
handles.append(plt.Line2D([0], [0], color='red', linestyle='--'))
labels.append('α = 0.05')
ax.legend(handles, labels, title='Effect Size', fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

# Panel B: sLNM Map Similarity
ax = axes[1]
for i, effect in enumerate(effect_sizes):
    offset = (i - 1) * width
    sims_cut = lnm_sim_all[effect][mask]
    sem_cut  = lnm_sem_all[effect][mask]
    ax.bar(x + offset, sims_cut, width, label=f'{effect:.2f}',
           color=colors[i], edgecolor='black', linewidth=0.5,
           yerr=sem_cut, capsize=2, error_kw={'elinewidth': 1})
ax.plot(x, bin_centers_cut, 'k--', alpha=0.5, label='x=y')
ax.set_xlabel('Spatial r of Ground Truth Network', fontsize=LABEL_SIZE)
ax.set_ylabel('Spatial r of sLNM Map', fontsize=LABEL_SIZE)
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, fontsize=TICK_SIZE, rotation=45, ha='right')
ax.tick_params(axis='y', labelsize=TICK_SIZE)
ax.set_ylim(-1, 1)
handles = [plt.Rectangle((0,0),1,1, facecolor=c, edgecolor='black', linewidth=0.5) for c in colors]
labels  = ['0.00', '0.30', '0.99']
handles.append(plt.Line2D([0], [0], color='black', linestyle='--', alpha=0.5))
labels.append('x=y')
ax.legend(handles, labels, title='Effect Size', fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

# Panel C: QQ Plot
ax = axes[2]
for i, effect in enumerate(effect_sizes):
    with open(target_files[effect], 'rb') as f:
        data = pickle.load(f)
    obs_p = np.sort(data['p_values'])
    exp_p = np.linspace(0, 1, len(obs_p))
    ax.scatter(exp_p, obs_p, alpha=0.6, s=20, color=colors[i], label=f'{effect:.2f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('Expected P-values', fontsize=LABEL_SIZE)
ax.set_ylabel('Observed P-values', fontsize=LABEL_SIZE)
ax.tick_params(axis='both', labelsize=TICK_SIZE)
handles = [plt.scatter([], [], color=c, s=20) for c in colors]
labels  = ['0.00', '0.30', '0.99']
handles.append(plt.Line2D([0], [0], color='black', linestyle='--', alpha=0.5))
labels.append('x=y')
ax.legend(handles, labels, title='Effect Size', fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

plt.tight_layout()
plt.show()